####Ex 1→ Upload and read BigMart CSV 

In [ ]:
from pyspark.sql.types import *
'''
Step 1→ import data to the same workspace folder 
        and copy its path
'''

# Step 2→ Define schema for all 12 columns you have seen in csv file
schema = StructType([
    StructField("Item_Identifier",         StringType(),  True),
    StructField("Item_Weight",             DoubleType(),  True),
    StructField("Item_Fat_Content",        StringType(),  True),
    StructField("Item_Visibility",         DoubleType(),  True),
    StructField("Item_Type",               StringType(),  True),
    StructField("Item_MRP",                DoubleType(),  True),
    StructField("Outlet_Identifier",       StringType(),  True),
    StructField("Outlet_Establishment_Year", IntegerType(), True),
    StructField("Outlet_Size",             StringType(),  True),
    StructField("Outlet_Location_Type",    StringType(),  True),
    StructField("Outlet_Type",             StringType(),  True),
    StructField("Item_Outlet_Sales",       DoubleType(),  True),
])

# Step 3→ Read CSV
df = spark.read.format("csv") \
         .option("header", True) \
         .option("sep", ",") \
         .schema(schema) \
         .load("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv")

print("Rows:", df.count())
print("Columns:", len(df.columns))
display(df)
df.printSchema()

# display(dbutils.fs.ls("dbfs:/Workspace/Pyspark_course/BigMart_Sales.csv"))

####Ex 2→ Explore the data- nulls,distinct values,stats
Before writing any pipeline, always profile the data. Find nulls, understand cardinality, and check numeric distributions. This is what every DE does on day one with a new dataset.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#1 Summary stats for all numeric columns
df.describe("Item_Weight","Item_Visibility","Item_MRP","Item_Outlet_Sales").show()

#2 Count null per column
null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
display(null_counts)

#3 Distinct values in categorical columns
display(df.select("Item_Fat_Content").distinct().collect())
display(df.select("Outlet_Type").distinct().collect())
display(df.select("Outlet_Size").distinct().collect())
display(df.select("Outlet_Location_Type").distinct().collect())

#4 How many unique items and outlets ?
print(f"Unique Items: {df.select("Item_Identifier").distinct().count()}")
print(f"Unique Outlets: {df.select("Outlet_Identifier").distinct().count()}")

# 5. Check Item_Type distribution
display(df.groupBy("Item_Type").count()   .orderBy(col("count").desc()))

####Ex 3→ Read only specific columns and filter on read
In production, never read all columns if you only need a few. Select and filter early — Spark pushes these down to the file scan level, saving memory and time.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# read only columns you need - projection pushdown
df_slim = spark.read.format("csv") \
                    .option("header",True) \
                    .load("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv") \
                    .select(
                    "Item_Identifier",
                    "Item_Type",
                    "Item_MRP",
                    "Outlet_Identifier",
                    "Outlet_Type",
                    "Item_Outlet_Sales")
display(df_slim)

# Filter immediately after read - only supermarkets
df_super = spark.read.format("csv") \
                     .option("header",True) \
                     .load("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv") \
                     .filter(col("Outlet_Type").contains("Supermarket"))
display(df_super)
    
# Filter Tier 1 locations with MRP > 200
df_premium = spark.read.format("csv") \
                       .option("header", "true") \
                       .load("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv") \
                       .filter((col("Outlet_Location_Type").contains("Tier 1")) & (col("Item_MRP").cast("double") > 200))
display(df_premium)

####Ex 4→ Write BigMart data as Parquet - with partitioning
Write the cleaned BigMart data as partitioned Parquet. Partitioning by Outlet_Type means any downstream query filtering by outlet type will only read the relevant folder.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

schema = StructType([
    StructField("Item_Identifier",          StringType(),  True),
    StructField("Item_Weight",              DoubleType(),  True),
    StructField("Item_Fat_Content",         StringType(),  True),
    StructField("Item_Visibility",          DoubleType(),  True),
    StructField("Item_Type",                StringType(),  True),
    StructField("Item_MRP",                 DoubleType(),  True),
    StructField("Outlet_Identifier",        StringType(),  True),
    StructField("Outlet_Establishment_Year",IntegerType(), True),
    StructField("Outlet_Size",              StringType(),  True),
    StructField("Outlet_Location_Type",     StringType(),  True),
    StructField("Outlet_Type",              StringType(),  True),
    StructField("Item_Outlet_Sales",        DoubleType(),  True),
])

df = spark.read.format("csv") \
               .option("header", True) \
               .option("sep", ",") \
               .schema(schema) \
               .csv("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv")
display(df)

# write partitioned Parquet by Outlet_Type
df.write.format("parquet") \
        .mode("overwrite") \
        .partitionBy("Outlet_Type") \
        .parquet("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Parquet")

# Inspect partition folder created
for f in dbutils.fs.ls("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Parquet"):
    print(f.path)

# Read back with partitioning pruning - only Grocery Store data
df_grocery = spark.read.format("parquet") \
                       .load("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Parquet") \
                       .filter(col("Outlet_Type").contains("Grocery Store")) 
                       
display(df_grocery)              

# Grocery Store rows
display(f"Grocery Store Rows: {df_grocery.count()}")           

# Check query plan -confirms partition pruning
df_grocery.explain()

####Ex 5→ Write modes -overwrite, append, Ignore on BigMart
Practice all four write modes using slices of the BigMart dataset. Understanding mode behaviour prevents accidental data loss or duplication in production.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Load full dataset
df = spark.read.format("csv") \
               .option("header",True) \
               .csv("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv")
display(df)

# Split into two batches to simulate incremental loads
df_tier1 = df.filter(col("Outlet_Location_Type").contains("Tier 1"))
df_tier2 = df.filter(col("Outlet_Location_Type").contains("Tier 2"))
df_tier3 = df.filter(col("Outlet_Location_Type").contains("Tier 3"))

print(f"Tier 1 rows: {df_tier1.count()}")
print(f"Tier 2 rows: {df_tier2.count()}")
print(f"Tier 3 rows: {df_tier3.count()}")

# overwrite - fresh write every time
df_tier1.write.format("parquet") \
              .mode("overwrite") \
              .parquet("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo")

print(f"After overwrite (tier 1): {spark.read.parquet('/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo').count()}")

# append - add tier2 on top of tier1
df_tier2.write.format("parquet") \
              .mode("append") \
              .parquet("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo")

print(f"After append (tier 2): {spark.read.parquet('/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo').count()}")

# To verfiy the append we have done
display(spark.read.parquet('/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo'))

# ignore - skips write since path already exists
df_tier3.write.format("parquet") \
              .mode("ignore") \
              .parquet("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo")

print(f"After ignore (tier 3): {spark.read.parquet('/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo').count()}")

# errorIfExists - throws error
try:
    df_tier3.write.format("parquet") \
                  .mode("errorIfExists") \
                  .parquet("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart_demo")
except Exception as e:
    print(f"Error caught: {type(e).__name__}")

####Ex 6→ Write as CSV and JSON - for downstream exports
Sometimes you need to export data back to CSV or JSON for downstream tools, reports, or APIs. Use coalesce(1) only for small export files.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# load data
df = spark.read.format("csv") \
               .option("header",True) \
               .csv("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv")
display(df)

# Build a summary: avg sales and avg mrp by Item_Type
summary_df = df.groupBy("Item_Type") \
               .agg(
                    round(avg(col("Item_Outlet_Sales").cast("double")), 2).alias("Avg_Sales"),
                    round(avg(col("Item_MRP").cast("double")), 2).alias("Avg_MRP")
               )
display(summary_df)

# Export to CSV (single file - small data)
summary_df.coalesce(1)\
          .write.mode("overwrite") \
          .option("header",True) \
          .csv("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Summary.csv")

# verify (CSV)
a = spark.read.csv("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Summary.csv")
display(a)

# Export to JSON
summary_df.coalesce(1)\
          .write.mode("overwrite") \
          .option("header",True) \
          .json("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Summary1.json")

# verify (JSON)
b = spark.read.json("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Summary1.json")
display(b)

####Ex 7→ Write BigMart as Delta Lake managed table
Write the full BigMart dataset as a Delta managed table — the standard storage pattern on Databricks. Once saved as a table, it's queryable from SQL, notebooks, and jobs.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

schema = StructType([
    StructField("Item_Identifier",          StringType(),  True),
    StructField("Item_Weight",              DoubleType(),  True),
    StructField("Item_Fat_Content",         StringType(),  True),
    StructField("Item_Visibility",          DoubleType(),  True),
    StructField("Item_Type",                StringType(),  True),
    StructField("Item_MRP",                 DoubleType(),  True),
    StructField("Outlet_Identifier",        StringType(),  True),
    StructField("Outlet_Establishment_Year",IntegerType(), True),
    StructField("Outlet_Size",              StringType(),  True),
    StructField("Outlet_Location_Type",     StringType(),  True),
    StructField("Outlet_Type",              StringType(),  True),
    StructField("Item_Outlet_Sales",        DoubleType(),  True),
])

df1 = spark.read.format("csv") \
               .option("header",True) \
               .schema(schema) \
               .csv("/Volumes/pyspark_course/dataset_for_pyspark_course/dataset/BigMart Sales.csv")
display(df1)

# write as Delta managed table
df1.write.format("delta") \
         .mode("overwrite") \
         .saveAsTable("BigMart_Sales_Delta")

# verify (can be in both ways given below)
print("Python way...")
display(spark.sql("SELECT * FROM BigMart_Sales_delta"))

print("SQL way...")
display(spark.table("BigMart_Sales_delta"))

# Query like SQL immediately
display(spark.sql("""
          SELECT Outlet_Type,
                 COUNT(*) AS num_items,
                 ROUND(AVG(Item_Outlet_Sales),2) AS avg_sales
          FROM BigMart_Sales_delta
          GROUP BY Outlet_Type
          ORDER BY avg_sales DESC
"""))

# Check table meta data
display(spark.sql("DESCRIBE DETAIL BigMart_Sales_delta") \
     .select("format","numFiles","sizeInBytes","location"))

####Ex 8→ Delta MERGE - upsert new outlet sales data
Simulate an incremental load: a new batch of BigMart sales arrives with updated records for existing items and a few new ones. MERGE handles both in one atomic operation.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

# Make sure BigMart_Sales_Delta table exists (run Ex-7)

schema = StructType([
    StructField("Item_Identifier",          StringType(),  True),
    StructField("Item_Weight",              DoubleType(),  True),
    StructField("Item_Fat_Content",         StringType(),  True),
    StructField("Item_Visibility",          DoubleType(),  True),
    StructField("Item_Type",                StringType(),  True),
    StructField("Item_MRP",                 DoubleType(),  True),
    StructField("Outlet_Identifier",        StringType(),  True),
    StructField("Outlet_Establishment_Year",IntegerType(), True),
    StructField("Outlet_Size",              StringType(),  True),
    StructField("Outlet_Location_Type",     StringType(),  True),
    StructField("Outlet_Type",              StringType(),  True),
    StructField("Item_Outlet_Sales",        DoubleType(),  True),
])

# Simulate incoming batch: 3 updated sales + 2 new items
incoming_data = [
    # existing item in OUT049 — sales updated
    ("FDA15", 9.3,  "Low Fat", 0.016, "Dairy",
     249.81, "OUT049", 1999, "Medium", "Tier 1",
     "Supermarket Type1", 4200.00),
    # existing item in OUT018 — sales updated
    ("DRC01", 5.92, "Regular", 0.019, "Soft Drinks",
     48.27,  "OUT018", 2009, "Medium", "Tier 3",
     "Supermarket Type2", 520.00),
    # brand new item never seen before
    ("NEW01", 10.0, "Low Fat", 0.010, "Dairy",
     199.99, "OUT049", 1999, "Medium", "Tier 1",
     "Supermarket Type1", 1500.00),
    ("NEW02", 7.5,  "Regular", 0.008, "Snack Foods",
     89.99,  "OUT018", 2009, "Medium", "Tier 3",
     "Supermarket Type2", 900.00),
]

df_incoming = spark.createDataFrame(incoming_data, schema)
display(df_incoming)

# MERGE into Delta table
delta_tb1 = DeltaTable.forName(spark, "BigMart_Sales_Delta")

delta_tb1.alias("target").merge(
    df_incoming.alias("source"),
    "target.Item_Identifier = source.Item_Identifier AND "
    "target.Outlet_Identifier = source.Outlet_Identifier"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

# verifying the MERGE
display(spark.sql("SELECT * FROM BigMart_Sales_delta"))
display(spark.sql("SELECT COUNT(*) FROM BigMart_Sales_Delta"))

# Verify the updates and inserts
spark.sql("""
    SELECT Item_Identifier, Outlet_Identifier, Item_Outlet_Sales
    FROM BigMart_Sales_Delta
    WHERE Item_Identifier IN ('FDA15','DRC01','NEW01','NEW02')
    ORDER BY Item_Identifier
""").show()

####Ex 9→ Delta time travel - explore BigMart table history
Since we ran overwrite and MERGE on the bigmart_sales table, it now has multiple versions. Use time travel to query previous states and restore if needed.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

# View the full change history of BigMart_Sales_Delta
display(
    spark.sql("DESCRIBE HISTORY bigmart_sales_delta")
        .select("version", "timestamp", "operation", "operationMetrics")
)

# Read version 0 - original CSV load
df_v0 = spark.read.format("delta") \
                  .option("versionAsOf",0) \
                  .table("BigMart_Sales_Delta")

display(f"Version 0 have: {df_v0.count()}")
display(f"Latest version have : {spark.table("BigMart_Sales_Delta").count()}")